# 📌 Fine-Tuning BERT on IMDB Dataset

## 🧠 Objective
The objective of this project is to build a text classification model using a pre-trained BERT model and fine-tune it on the IMDB movie reviews dataset for sentiment analysis.

This task helps in understanding how transformer-based models like BERT can be adapted for real-world NLP problems.

---

## 📚 About Dataset
The IMDB dataset consists of movie reviews labeled as:
- **Positive (1)**
- **Negative (0)**

It is widely used for sentiment analysis tasks.

---



## 🔄 Workflow
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Comparison

---

## 🎯 Learning Outcomes
- Understand BERT architecture for text classification  
- Learn tokenization using pre-trained models  
- Perform fine-tuning on a transformer model  
- Evaluate model using multiple performance metrics  
- Compare different training strategies  

---


In [4]:
# CLEAN INSTALL (no conflicts)
!pip install transformers==4.37.2 datasets scikit-learn -q

import torch
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# IMPORTANT: avoid Trainer (causing your error)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from torch.utils.data import DataLoader
from torch.optim import AdamW


In [5]:
# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# LOAD DATA
dataset = load_dataset("imdb")

# SMALL DATA (FAST)
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

# MODEL
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# TRAIN (FAST: 1 epoch)
model.train()
for epoch in range(1):
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

print("Training Done ✅")




Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Done ✅


In [6]:
# EVALUATION
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [7]:
# METRICS
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
acc = accuracy_score(all_labels, all_preds)

print("\nAccuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# CONFUSION MATRIX
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


Accuracy: 0.81
Precision: 0.912568306010929
Recall: 0.6788617886178862
F1 Score: 0.7785547785547785

Confusion Matrix:
[[238  16]
 [ 79 167]]


## 🧹 Data Preprocessing

In this step, the dataset is cleaned and prepared for training:
- Removed unnecessary noise (if any)
- Converted dataset into train, validation, and test sets
- Ensured labels are correctly mapped

A subset of the dataset is used to reduce training time while maintaining performance.

---


In [18]:
!pip install -q transformers datasets scikit-learn

import torch
import numpy as np
import re

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# LOAD DATA
dataset = load_dataset("imdb")

# =========================
# PREPROCESSING (ADDED)
# =========================
def clean_text(example):
    text = example["text"].lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    example["text"] = text
    return example

dataset = dataset.map(clean_text)


Device: cuda


## 🔤 Tokenization

We use the **bert-base-uncased tokenizer** to convert raw text into tokens that BERT understands.

Steps involved:
- Convert text into token IDs
- Add special tokens ([CLS], [SEP])
- Apply padding and truncation
- Generate attention masks

This ensures all inputs are in a uniform format for the model.


## 🤖 Model Training

We use a pre-trained BERT model:
- `AutoModelForSequenceClassification`

Training details:
- Optimizer: AdamW
- Learning Rate: 2e-5
- Loss Function: Cross-Entropy Loss

The model is trained on labeled IMDB reviews to learn sentiment classification.


## 🧪 Experiments

Two experiments were conducted:

### 🔹 Experiment 1: Frozen BERT
- All BERT layers are frozen
- Only classification head is trained

### 🔹 Experiment 2: Fine-Tuning Last Layers
- Last 2 layers of BERT are unfrozen
- Allows deeper learning and better performance

### 📊 Observation
Fine-tuning improves performance compared to freezing, as the model adapts better to the dataset.


In [16]:
# =========================
# SMALL DATA (FAST)
# =========================
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
temp_data = dataset["test"].shuffle(seed=42).select(range(1000))

# =========================
# VALIDATION SPLIT (ADDED)
# =========================
temp_texts = list(temp_data["text"])
temp_labels = list(temp_data["label"])

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

# Convert back to dataset format
from datasets import Dataset

val_data = Dataset.from_dict({"text": val_texts, "label": val_labels})
test_data = Dataset.from_dict({"text": test_texts, "label": test_labels})

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize, batched=True)
val_data = val_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)
test_loader = DataLoader(test_data, batch_size=16)

# =========================
# TRAIN FUNCTION
# =========================
def train_model(model):
    optimizer = AdamW(model.parameters(), lr=2e-5)
    model.train()

    for epoch in range(1):
        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

    return model

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate(model):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    acc = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    return acc, precision, recall, f1, cm

# =========================
# 🔬 EXPERIMENT 1: Frozen BERT
# =========================
model1 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

for param in model1.bert.parameters():
    param.requires_grad = False

model1 = train_model(model1)
acc1, p1, r1, f1_1, cm1 = evaluate(model1)

print("\n=== Frozen BERT ===")
print("Accuracy:", acc1)
print("Precision:", p1)
print("Recall:", r1)
print("F1 Score:", f1_1)
print("Confusion Matrix:\n", cm1)

# =========================
# 🔬 EXPERIMENT 2: Fine-tune last 2 layers
# =========================
model2 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

for param in model2.bert.parameters():
    param.requires_grad = False

for param in model2.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

model2 = train_model(model2)
acc2, p2, r2, f2, cm2 = evaluate(model2)

print("\n=== Fine-tuned Last 2 Layers ===")
print("Accuracy:", acc2)
print("Precision:", p2)
print("Recall:", r2)
print("F1 Score:", f2)
print("Confusion Matrix:\n", cm2)

# =========================
# 📊 COMPARISON (ADDED)
# =========================
print("\n=== FINAL COMPARISON ===")
print("Frozen Accuracy:", acc1)
print("Fine-tuned Accuracy:", acc2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Frozen BERT ===
Accuracy: 0.528
Precision: 0.519434628975265
Recall: 0.5951417004048583
F1 Score: 0.5547169811320755
Confusion Matrix:
 [[117 136]
 [100 147]]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Fine-tuned Last 2 Layers ===
Accuracy: 0.792
Precision: 0.7871485943775101
Recall: 0.7935222672064778
F1 Score: 0.7903225806451613
Confusion Matrix:
 [[200  53]
 [ 51 196]]

=== FINAL COMPARISON ===
Frozen Accuracy: 0.528
Fine-tuned Accuracy: 0.792


## 📊 Model Evaluation

The model is evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix

These metrics help in understanding model performance from multiple perspectives.

---

## 🏁 Conclusion

In this project, we successfully fine-tuned a BERT model for sentiment analysis using the IMDB dataset.

### ✅ Key Achievements:
- Implemented BERT-based text classification
- Performed tokenization using pre-trained models
- Conducted experiments with different training strategies
- Evaluated model using multiple metrics

### 📌 Insights:
- Fine-tuning BERT layers improves performance significantly
- Transformer models outperform traditional NLP approaches

---

